## Clean up
Please make sure to comment the below section if you are planning to use the Knowledge Base that you created above for building your RAG application.
If you only wanted to try out creating the KB using SDK, then please make sure to delete all the resources that were created as you will be incurred cost for storing documents in OSS index.

#### Delete KnowledgeBase and delete resources after completing all the notebooks.


In [1]:
%store -r

AttributeError: 'PickleShareDB' object has no attribute 'keys'

In [2]:
import boto3

In [3]:
boto3_session = boto3.Session()
bedrock_agent_client = boto3_session.client('bedrock-agent', region_name=boto3_session.region_name)
aoss_client = boto3.client('opensearchserverless')
s3_client = boto3_session.client('s3', region_name=boto3_session.region_name)
iam_client = boto3.client("iam")

#### delete Bedrock KnowledgeBase data sources

In [5]:
kb_id = "DSXJFKGEHW"

response = bedrock_agent_client.list_data_sources(
    knowledgeBaseId=kb_id,
)
data_source_ids = [ x['dataSourceId'] for x in response['dataSourceSummaries']]

for data_source_id in data_source_ids:
    bedrock_agent_client.delete_data_source(dataSourceId = data_source_id, knowledgeBaseId=kb_id)

#### Remove KnowledgeBases and OpenSearch Collection

In [6]:
response = bedrock_agent_client.get_knowledge_base(knowledgeBaseId=kb_id)

In [7]:
kb_role_name = response['knowledgeBase']['roleArn'].split("/")[-1]

In [8]:
kb_attached_role_policies_response = iam_client.list_attached_role_policies(
    RoleName=kb_role_name)

In [9]:
kb_attached_role_policies = kb_attached_role_policies_response['AttachedPolicies']

In [11]:
collection = "m7kpi73ougi3i428det9"
bedrock_agent_client.delete_knowledge_base(knowledgeBaseId=kb_id)
aoss_client.delete_collection(id=collection['createCollectionDetail']['id'])
aoss_client.delete_access_policy(type="data", name=access_policy['accessPolicyDetail']['name'])
aoss_client.delete_security_policy(type="network", name=network_policy['securityPolicyDetail']['name'])
aoss_client.delete_security_policy(type="encryption", name=encryption_policy['securityPolicyDetail']['name'])

ResourceNotFoundException: An error occurred (ResourceNotFoundException) when calling the DeleteKnowledgeBase operation: KnowledgeBase with id DSXJFKGEHW is not found.

#### Delete role and policies


In [12]:
for policy in kb_attached_role_policies:
    iam_client.detach_role_policy(
            RoleName=kb_role_name,
            PolicyArn=policy['PolicyArn']
    )

In [13]:
iam_client.delete_role(RoleName=kb_role_name)

{'ResponseMetadata': {'RequestId': '8fec0f41-7eb3-4ddf-bdc0-fd13f3ae00d5',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 15 Sep 2024 07:58:54 GMT',
   'x-amzn-requestid': '8fec0f41-7eb3-4ddf-bdc0-fd13f3ae00d5',
   'content-type': 'text/xml',
   'content-length': '200'},
  'RetryAttempts': 0}}

In [14]:
for policy in kb_attached_role_policies:
    iam_client.delete_policy(PolicyArn=policy['PolicyArn'])

ClientError: An error occurred (AccessDenied) when calling the DeletePolicy operation: User: arn:aws:iam::137360334857:user/farzana.anjum@capgemini.com is not authorized to perform: iam:DeletePolicy on resource: policy arn:aws:iam::137360334857:policy/AmazonBedrockS3PolicyForKnowledgeBase_368 because no identity-based policy allows the iam:DeletePolicy action

#### Delete S3 objects

In [16]:
bucket_name = "bedrock-kb-eu-central-1-137360334857"
objects = s3_client.list_objects(Bucket=bucket_name)
if 'Contents' in objects:
    for obj in objects['Contents']:
        s3_client.delete_object(Bucket=bucket_name, Key=obj['Key'])
s3_client.delete_bucket(Bucket=bucket_name)

{'ResponseMetadata': {'RequestId': '5TMVNYFV6TKVAQ6J',
  'HostId': 'KNiQI6dy++045mfeL3CnDdVa9yR+Q2rGTEpmMC1vwob6+ZVCfBDuCpQipkV78YS5t28rWtV/rPOuViomC9EUKRgaYKHV6CgAUiuwZq6k9Iw=',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'x-amz-id-2': 'KNiQI6dy++045mfeL3CnDdVa9yR+Q2rGTEpmMC1vwob6+ZVCfBDuCpQipkV78YS5t28rWtV/rPOuViomC9EUKRgaYKHV6CgAUiuwZq6k9Iw=',
   'x-amz-request-id': '5TMVNYFV6TKVAQ6J',
   'date': 'Sun, 15 Sep 2024 07:59:53 GMT',
   'server': 'AmazonS3'},
  'RetryAttempts': 0}}